In [0]:
# Imports necessários para esta célula
import shutil
import zipfile
from pathlib import Path

In [0]:
# Obtém o catálogo e schema atuais para usar como valores padrão dos widgets.
CURRENT_CATALOG = spark.sql("SELECT current_catalog() AS catalog").first()["catalog"]
CURRENT_SCHEMA = spark.sql("SELECT current_schema() AS schema").first()["schema"]


def ensure_text_widget(name: str, default_value: str, label: str) -> None:
    """Cria um widget somente se ele ainda não existir."""
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default_value, label)


def ensure_dropdown_widget(name: str, default_value: str, choices: list[str], label: str) -> None:
    """Cria um dropdown somente se ele ainda não existir."""
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default_value, choices, label)


ensure_text_widget("catalogo", CURRENT_CATALOG, "01 - Catálogo Unity Catalog")
ensure_text_widget("schema", CURRENT_SCHEMA, "02 - Schema")
ensure_text_widget("volume", "cnpj_brasil", "03 - Volume")
ensure_text_widget("pasta_projeto", "CNPJ_BRASIL", "04 - Pasta do projeto no volume")
ensure_text_widget("periodo", "auto", "05 - Período YYYY-MM ou auto")
ensure_dropdown_widget("sobrescrever_raw", "false", ["false", "true"], "06 - Sobrescrever ZIPs existentes")
ensure_dropdown_widget("limpar_extraidos", "true", ["true", "false"], "07 - Limpar EXTRACTED antes de extrair")
ensure_dropdown_widget("apagar_raw", "false", ["false", "true"], "08 - Apagar RAW após extração")
ensure_dropdown_widget("carregar_delta", "true", ["true", "false"], "09 - Carregar tabelas Delta")

print("Widgets preparados. Ajuste os valores no topo do notebook e execute as células seguintes.")

Widgets preparados. Ajuste os valores no topo do notebook e execute as células seguintes.


In [0]:
import os
import re
import shutil
import tempfile
import zipfile
import requests
import xml.etree.ElementTree as ET

from datetime import datetime
from pathlib import Path
from urllib.parse import quote, unquote

from pyspark.sql import functions as F
from pyspark.sql.types import StructField, StructType, StringType


def as_bool(value: str) -> bool:
    return value.strip().lower() in {"1", "true", "sim", "yes", "y"}


def validate_identifier(value: str, field_name: str) -> str:
    """Restringe identificadores para evitar nomes inválidos e SQL injection."""
    value = value.strip()
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
        raise ValueError(
            f"{field_name} inválido: {value!r}. Use letras, números e underscore, "
            "sem espaços ou caracteres especiais."
        )
    return value


CATALOG = validate_identifier(dbutils.widgets.get("catalogo"), "Catálogo")
SCHEMA = validate_identifier(dbutils.widgets.get("schema"), "Schema")
VOLUME = validate_identifier(dbutils.widgets.get("volume"), "Volume")
PROJECT_FOLDER = validate_identifier(dbutils.widgets.get("pasta_projeto"), "Pasta do projeto")
PERIODO_PARAM = dbutils.widgets.get("periodo").strip()
OVERWRITE_RAW = as_bool(dbutils.widgets.get("sobrescrever_raw"))
CLEAN_EXTRACTED = as_bool(dbutils.widgets.get("limpar_extraidos"))
DELETE_RAW = as_bool(dbutils.widgets.get("apagar_raw"))
LOAD_DELTA = as_bool(dbutils.widgets.get("carregar_delta"))

if CATALOG.lower() in {"hive_metastore", "spark_catalog"}:
    raise ValueError(
        "Este notebook usa Unity Catalog Volumes. Selecione um catálogo Unity Catalog "
        "no widget 'catalogo', em vez de hive_metastore/spark_catalog."
    )

BASE_URL = "https://arquivos.receitafederal.gov.br"
SHARE_TOKEN = "YggdBLfdninEJX9"  # token público da pasta compartilhada da Receita Federal

TARGET_ENTITIES = [
    "Empresas",
    "Estabelecimentos",
    "Socios",
    "Municipios",
    "Naturezas",
]

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
PROJECT_ROOT = f"{VOLUME_ROOT}/{PROJECT_FOLDER}"
RAW_ROOT = f"{PROJECT_ROOT}/RAW"
EXTRACTED_ROOT = f"{PROJECT_ROOT}/EXTRACTED"

# O diretório local é efêmero e existe apenas durante a vida do compute.
LOCAL_TMP_ROOT = Path("/local_disk0/tmp/cnpj_brasil")
try:
    LOCAL_TMP_ROOT.mkdir(parents=True, exist_ok=True)
except OSError:
    LOCAL_TMP_ROOT = Path(tempfile.gettempdir()) / "cnpj_brasil"
    LOCAL_TMP_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Catálogo/schema: {CATALOG}.{SCHEMA}")
print(f"Volume: {VOLUME_ROOT}")
print(f"Projeto: {PROJECT_ROOT}")
print(f"Buffer local: {LOCAL_TMP_ROOT}")

Catálogo/schema: workspace.default
Volume: /Volumes/workspace/default/cnpj_brasil
Projeto: /Volumes/workspace/default/cnpj_brasil/CNPJ_BRASIL
Buffer local: /local_disk0/tmp/cnpj_brasil


In [0]:
import os
import re
import shutil
import tempfile
import zipfile
import requests
import xml.etree.ElementTree as ET

from datetime import datetime
from pathlib import Path
from urllib.parse import quote, unquote

from pyspark.sql import functions as F
from pyspark.sql.types import StructField, StructType, StringType


def as_bool(value: str) -> bool:
    return value.strip().lower() in {"1", "true", "sim", "yes", "y"}


def validate_identifier(value: str, field_name: str) -> str:
    """Restringe identificadores para evitar nomes inválidos e SQL injection."""
    value = value.strip()
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
        raise ValueError(
            f"{field_name} inválido: {value!r}. Use letras, números e underscore, "
            "sem espaços ou caracteres especiais."
        )
    return value


CATALOG = validate_identifier(dbutils.widgets.get("catalogo"), "Catálogo")
SCHEMA = validate_identifier(dbutils.widgets.get("schema"), "Schema")
VOLUME = validate_identifier(dbutils.widgets.get("volume"), "Volume")
PROJECT_FOLDER = validate_identifier(dbutils.widgets.get("pasta_projeto"), "Pasta do projeto")
PERIODO_PARAM = dbutils.widgets.get("periodo").strip()
OVERWRITE_RAW = as_bool(dbutils.widgets.get("sobrescrever_raw"))
CLEAN_EXTRACTED = as_bool(dbutils.widgets.get("limpar_extraidos"))
DELETE_RAW = as_bool(dbutils.widgets.get("apagar_raw"))
LOAD_DELTA = as_bool(dbutils.widgets.get("carregar_delta"))

if CATALOG.lower() in {"hive_metastore", "spark_catalog"}:
    raise ValueError(
        "Este notebook usa Unity Catalog Volumes. Selecione um catálogo Unity Catalog "
        "no widget 'catalogo', em vez de hive_metastore/spark_catalog."
    )

BASE_URL = "https://arquivos.receitafederal.gov.br"
SHARE_TOKEN = "YggdBLfdninEJX9"  # token público da pasta compartilhada da Receita Federal

TARGET_ENTITIES = [
    "Empresas",
    "Estabelecimentos",
    "Socios",
    "Municipios",
    "Naturezas",
]

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
PROJECT_ROOT = f"{VOLUME_ROOT}/{PROJECT_FOLDER}"
RAW_ROOT = f"{PROJECT_ROOT}/RAW"
EXTRACTED_ROOT = f"{PROJECT_ROOT}/EXTRACTED"

# O diretório local é efêmero e existe apenas durante a vida do compute.
LOCAL_TMP_ROOT = Path("/local_disk0/tmp/cnpj_brasil")
try:
    LOCAL_TMP_ROOT.mkdir(parents=True, exist_ok=True)
except OSError:
    LOCAL_TMP_ROOT = Path(tempfile.gettempdir()) / "cnpj_brasil"
    LOCAL_TMP_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Catálogo/schema: {CATALOG}.{SCHEMA}")
print(f"Volume: {VOLUME_ROOT}")
print(f"Projeto: {PROJECT_ROOT}")
print(f"Buffer local: {LOCAL_TMP_ROOT}")

Catálogo/schema: workspace.default
Volume: /Volumes/workspace/default/cnpj_brasil
Projeto: /Volumes/workspace/default/cnpj_brasil/CNPJ_BRASIL
Buffer local: /local_disk0/tmp/cnpj_brasil


In [0]:
# Cria o schema e o volume gerenciado, quando o usuário tiver permissão.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")

try:
    spark.sql(
        f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME}` "
        "COMMENT 'Arquivos brutos e extraídos do pipeline CNPJ Brasil'"
    )
except Exception as exc:
    print(f"[AVISO] Não foi possível criar o volume automaticamente: {exc}")
    print("[INFO] O notebook continuará caso o volume já exista e você possua USE/READ/WRITE VOLUME.")

try:
    os.makedirs(RAW_ROOT, exist_ok=True)
    os.makedirs(EXTRACTED_ROOT, exist_ok=True)
except Exception as exc:
    raise RuntimeError(
        f"Não foi possível acessar/criar pastas em {VOLUME_ROOT}. "
        "Confirme a existência do volume e suas permissões."
    ) from exc

print("[OK] Estrutura do volume preparada.")

[OK] Estrutura do volume preparada.


In [0]:
def recent_periods(quantity: int = 4) -> list[str]:
    """Retorna o mês atual e os meses anteriores no formato YYYY-MM."""
    now = datetime.now()
    periods = []
    year, month = now.year, now.month

    for offset in range(quantity):
        absolute_month = year * 12 + (month - 1) - offset
        candidate_year, candidate_month_zero = divmod(absolute_month, 12)
        periods.append(f"{candidate_year:04d}-{candidate_month_zero + 1:02d}")

    return periods


def list_remote_files(token: str, period: str, target_entities: list[str]) -> list[tuple[str, str]]:
    """Lista ZIPs do período via WebDAV, com fallback para a página compartilhada."""
    webdav_url = f"{BASE_URL}/public.php/webdav/{period}"
    filenames: set[str] = set()

    try:
        response = requests.request(
            "PROPFIND",
            webdav_url,
            auth=(token, ""),
            headers={"Depth": "1"},
            timeout=(30, 120),
        )

        if response.status_code in {200, 207}:
            root = ET.fromstring(response.content)
            for element in root.findall(".//{DAV:}href"):
                href = element.text or ""
                filename = unquote(href.rstrip("/").split("/")[-1]).strip()
                if filename.lower().endswith(".zip"):
                    filenames.add(filename)
        else:
            print(f"[AVISO] PROPFIND retornou HTTP {response.status_code} para {period}.")

    except Exception as exc:
        print(f"[AVISO] Falha na consulta WebDAV de {period}: {exc}")

    if not filenames:
        try:
            page_url = f"{BASE_URL}/index.php/s/{token}?dir=/{period}"
            response = requests.get(page_url, timeout=(30, 120))
            response.raise_for_status()
            for match in re.findall(r'href=["\']([^"\']+\.zip)["\']', response.text, flags=re.I):
                filenames.add(unquote(match.rstrip("/").split("/")[-1]).strip())
        except Exception as exc:
            print(f"[AVISO] Falha no fallback HTML de {period}: {exc}")

    selected: list[tuple[str, str]] = []
    for filename in sorted(filenames):
        for entity in target_entities:
            if entity.lower() in filename.lower():
                selected.append((filename, entity))
                break

    return selected


def discover_period_and_files(period_param: str) -> tuple[str, list[tuple[str, str]]]:
    """Usa o período informado ou procura automaticamente nos últimos quatro meses."""
    if period_param.lower() == "auto":
        candidates = recent_periods(4)
    else:
        if not re.fullmatch(r"\d{4}-\d{2}", period_param):
            raise ValueError("O widget 'periodo' deve ser 'auto' ou estar no formato YYYY-MM.")
        candidates = [period_param]

    core_entities = {"Empresas", "Estabelecimentos", "Socios"}
    best_result: tuple[str, list[tuple[str, str]]] | None = None
    best_coverage = -1

    for period in candidates:
        files = list_remote_files(SHARE_TOKEN, period, TARGET_ENTITIES)
        found_entities = {entity for _, entity in files}
        coverage = len(found_entities)
        print(f"[*] {period}: {len(files)} ZIPs; entidades encontradas: {sorted(found_entities)}")

        if coverage > best_coverage:
            best_result = (period, files)
            best_coverage = coverage

        if core_entities.issubset(found_entities):
            return period, files

    if best_result and best_result[1]:
        print("[AVISO] Nenhum período apresentou todas as entidades principais; usando o melhor resultado encontrado.")
        return best_result

    raise RuntimeError(
        "Nenhum arquivo foi encontrado. Verifique o período, o acesso à internet do compute "
        "e a disponibilidade da pasta da Receita Federal."
    )


def download_file(token: str, period: str, filename: str, destination: Path) -> None:
    """Baixa um ZIP para o disco local do driver usando escrita atômica."""
    encoded_filename = quote(filename)
    url = f"{BASE_URL}/public.php/webdav/{period}/{encoded_filename}"
    partial = destination.with_suffix(destination.suffix + ".part")

    destination.parent.mkdir(parents=True, exist_ok=True)
    if partial.exists():
        partial.unlink()

    with requests.get(url, auth=(token, ""), stream=True, timeout=(30, 1800)) as response:
        response.raise_for_status()
        with partial.open("wb") as output:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    output.write(chunk)

    if partial.stat().st_size == 0:
        partial.unlink(missing_ok=True)
        raise IOError(f"Download vazio: {filename}")

    partial.replace(destination)


def run_download() -> tuple[str, list[tuple[str, str]]]:
    period, files_to_process = discover_period_and_files(PERIODO_PARAM)
    print(f"\n[INFO] Período selecionado: {period}")

    successes = 0
    failures: list[str] = []

    for filename, entity in files_to_process:
        raw_dir = Path(f"{RAW_ROOT}/{period}/{entity}")
        raw_path = raw_dir / filename

        if raw_path.exists() and raw_path.stat().st_size > 0 and not OVERWRITE_RAW:
            print(f"[SKIP] Já existe no RAW: {entity}/{filename}")
            successes += 1
            continue

        local_path = LOCAL_TMP_ROOT / "download" / filename
        try:
            print(f"[*] Baixando {filename} -> RAW/{entity}")
            download_file(SHARE_TOKEN, period, filename, local_path)
            raw_dir.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(local_path, raw_path)
            size_mb = raw_path.stat().st_size / (1024 * 1024)
            print(f"[OK] {entity}/{filename} ({size_mb:,.2f} MB)")
            successes += 1
        except Exception as exc:
            failures.append(filename)
            print(f"[ERRO] {filename}: {exc}")
        finally:
            local_path.unlink(missing_ok=True)

    print(f"\n[RESUMO] Downloads disponíveis: {successes}; falhas: {len(failures)}")
    if failures:
        print(f"[AVISO] Arquivos com falha: {failures}")

    if successes == 0:
        raise RuntimeError("Nenhum arquivo ficou disponível na camada RAW.")

    return period, files_to_process


PERIODO_REFERENCIA, REMOTE_FILES = run_download()

# Caminhos ativos da competência selecionada.
# Estas variáveis são usadas nas etapas de extração, limpeza e carga Delta.
ACTIVE_RAW_ROOT = f"{RAW_ROOT}/{PERIODO_REFERENCIA}"
ACTIVE_EXTRACTED_ROOT = f"{EXTRACTED_ROOT}/{PERIODO_REFERENCIA}"

Path(ACTIVE_RAW_ROOT).mkdir(parents=True, exist_ok=True)
Path(ACTIVE_EXTRACTED_ROOT).mkdir(parents=True, exist_ok=True)

print(f"[OK] RAW ativo: {ACTIVE_RAW_ROOT}")
print(f"[OK] EXTRACTED ativo: {ACTIVE_EXTRACTED_ROOT}")

[AVISO] PROPFIND retornou HTTP 404 para 2026-08.
[*] 2026-08: 0 ZIPs; entidades encontradas: []
[*] 2026-07: 32 ZIPs; entidades encontradas: ['Empresas', 'Estabelecimentos', 'Municipios', 'Naturezas', 'Socios']

[INFO] Período selecionado: 2026-07
[SKIP] Já existe no RAW: Empresas/Empresas0.zip
[SKIP] Já existe no RAW: Empresas/Empresas1.zip
[SKIP] Já existe no RAW: Empresas/Empresas2.zip
[SKIP] Já existe no RAW: Empresas/Empresas3.zip
[SKIP] Já existe no RAW: Empresas/Empresas4.zip
[SKIP] Já existe no RAW: Empresas/Empresas5.zip
[SKIP] Já existe no RAW: Empresas/Empresas6.zip
[SKIP] Já existe no RAW: Empresas/Empresas7.zip
[SKIP] Já existe no RAW: Empresas/Empresas8.zip
[SKIP] Já existe no RAW: Empresas/Empresas9.zip
[SKIP] Já existe no RAW: Estabelecimentos/Estabelecimentos0.zip
[SKIP] Já existe no RAW: Estabelecimentos/Estabelecimentos1.zip
[SKIP] Já existe no RAW: Estabelecimentos/Estabelecimentos2.zip
[SKIP] Já existe no RAW: Estabelecimentos/Estabelecimentos3.zip
[SKIP] Já existe

In [0]:
# Imports locais para permitir a execução isolada desta célula no Databricks
import shutil
import zipfile
from pathlib import Path

# Garante que a célula falhe com uma mensagem clara quando as etapas anteriores
# não tiverem sido executadas no compute atual.
_REQUIRED_CONTEXT = [
    "RAW_ROOT",
    "EXTRACTED_ROOT",
    "PERIODO_REFERENCIA",
    "LOCAL_TMP_ROOT",
    "TARGET_ENTITIES",
    "CLEAN_EXTRACTED",
]
_MISSING_CONTEXT = [name for name in _REQUIRED_CONTEXT if name not in globals()]
if _MISSING_CONTEXT:
    raise RuntimeError(
        "Contexto do pipeline não inicializado: "
        + ", ".join(_MISSING_CONTEXT)
        + ". Execute as seções 1, 2 e 3, ou use Run all above."
    )

# Fallback defensivo: reconstrói os caminhos ativos quando a competência já existe,
# mas as variáveis ACTIVE_* ainda não foram criadas na sessão.
ACTIVE_RAW_ROOT = globals().get(
    "ACTIVE_RAW_ROOT",
    f"{RAW_ROOT}/{PERIODO_REFERENCIA}",
)
ACTIVE_EXTRACTED_ROOT = globals().get(
    "ACTIVE_EXTRACTED_ROOT",
    f"{EXTRACTED_ROOT}/{PERIODO_REFERENCIA}",
)

Path(ACTIVE_RAW_ROOT).mkdir(parents=True, exist_ok=True)
Path(ACTIVE_EXTRACTED_ROOT).mkdir(parents=True, exist_ok=True)

print(f"[INFO] RAW ativo: {ACTIVE_RAW_ROOT}")
print(f"[INFO] EXTRACTED ativo: {ACTIVE_EXTRACTED_ROOT}")

def safe_extract(zip_file: zipfile.ZipFile, destination: Path) -> None:
    """Extrai o ZIP bloqueando entradas com path traversal."""
    destination_abs = destination.resolve()

    for member in zip_file.infolist():
        member_path = (destination / member.filename).resolve()
        if destination_abs != member_path and destination_abs not in member_path.parents:
            raise ValueError(f"Entrada insegura no ZIP: {member.filename}")

    zip_file.extractall(destination)


def list_zip_paths(entity: str) -> list[Path]:
    raw_dir = Path(f"{ACTIVE_RAW_ROOT}/{entity}")
    if not raw_dir.exists():
        return []
    return sorted(path for path in raw_dir.iterdir() if path.is_file() and path.suffix.lower() == ".zip")


def extract_entity(entity: str) -> int:
    zip_paths = list_zip_paths(entity)
    if not zip_paths:
        print(f"[AVISO] Nenhum ZIP encontrado em RAW/{entity}")
        return 0

    entity_output = Path(f"{ACTIVE_EXTRACTED_ROOT}/{entity}")
    entity_output.mkdir(parents=True, exist_ok=True)
    extracted_count = 0

    for volume_zip_path in zip_paths:
        local_entity_root = LOCAL_TMP_ROOT / "extract" / entity / volume_zip_path.stem
        local_zip_path = local_entity_root / volume_zip_path.name
        local_output = local_entity_root / "out"

        try:
            if local_entity_root.exists():
                shutil.rmtree(local_entity_root)
            local_output.mkdir(parents=True, exist_ok=True)

            print(f"[*] Extraindo {entity}/{volume_zip_path.name}")
            shutil.copyfile(volume_zip_path, local_zip_path)

            with zipfile.ZipFile(local_zip_path, "r") as zip_ref:
                safe_extract(zip_ref, local_output)

            for source_file in local_output.rglob("*"):
                if not source_file.is_file():
                    continue

                relative_path = source_file.relative_to(local_output)
                destination_file = entity_output / relative_path
                destination_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(source_file, destination_file)
                extracted_count += 1
                print(f"[OK] {entity}/{relative_path}")

        except zipfile.BadZipFile as exc:
            print(f"[ERRO] ZIP inválido {volume_zip_path.name}: {exc}")
        except Exception as exc:
            print(f"[ERRO] Falha ao extrair {volume_zip_path.name}: {exc}")
        finally:
            if local_entity_root.exists():
                shutil.rmtree(local_entity_root, ignore_errors=True)

    return extracted_count


def run_extraction() -> dict[str, int]:
    if CLEAN_EXTRACTED and Path(ACTIVE_EXTRACTED_ROOT).exists():
        print(f"[*] Limpando camada EXTRACTED: {ACTIVE_EXTRACTED_ROOT}")
        shutil.rmtree(ACTIVE_EXTRACTED_ROOT)

    for entity in TARGET_ENTITIES:
        Path(f"{ACTIVE_EXTRACTED_ROOT}/{entity}").mkdir(parents=True, exist_ok=True)

    result = {}
    for entity in TARGET_ENTITIES:
        print("\n" + "=" * 80)
        print(f"ENTIDADE: {entity}")
        print("=" * 80)
        result[entity] = extract_entity(entity)

    print("\n[RESUMO DA EXTRAÇÃO]")
    for entity, count in result.items():
        print(f"- {entity}: {count} arquivo(s) extraído(s)")

    return result


EXTRACTION_RESULT = run_extraction()

[INFO] RAW ativo: /Volumes/workspace/default/cnpj_brasil/CNPJ_BRASIL/RAW/2026-07
[INFO] EXTRACTED ativo: /Volumes/workspace/default/cnpj_brasil/CNPJ_BRASIL/EXTRACTED/2026-07
[*] Limpando camada EXTRACTED: /Volumes/workspace/default/cnpj_brasil/CNPJ_BRASIL/EXTRACTED/2026-07

ENTIDADE: Empresas
[*] Extraindo Empresas/Empresas0.zip
[OK] Empresas/K3241.K03200Y0.D60711.EMPRECSV
[*] Extraindo Empresas/Empresas1.zip
[OK] Empresas/K3241.K03200Y1.D60711.EMPRECSV
[*] Extraindo Empresas/Empresas2.zip
[OK] Empresas/K3241.K03200Y2.D60711.EMPRECSV
[*] Extraindo Empresas/Empresas3.zip
[OK] Empresas/K3241.K03200Y3.D60711.EMPRECSV
[*] Extraindo Empresas/Empresas4.zip
[OK] Empresas/K3241.K03200Y4.D60711.EMPRECSV
[*] Extraindo Empresas/Empresas5.zip
[OK] Empresas/K3241.K03200Y5.D60711.EMPRECSV
[*] Extraindo Empresas/Empresas6.zip
[OK] Empresas/K3241.K03200Y6.D60711.EMPRECSV
[*] Extraindo Empresas/Empresas7.zip
[OK] Empresas/K3241.K03200Y7.D60711.EMPRECSV
[*] Extraindo Empresas/Empresas8.zip
[OK] Empresas

## 5. Exclusão opcional da camada RAW

In [0]:
SCHEMAS = {
    "Empresas": StructType([
        StructField("CNPJ_BASICO", StringType(), True),
        StructField("RAZAO_SOCIAL", StringType(), True),
        StructField("NATUREZA_JURIDICA", StringType(), True),
        StructField("QUALIFICACAO_RESPONSAVEL", StringType(), True),
        StructField("CAPITAL_SOCIAL", StringType(), True),
        StructField("PORTE_EMPRESA", StringType(), True),
        StructField("ENTE_FEDERATIVO", StringType(), True),
    ]),
    "Estabelecimentos": StructType([
        StructField("CNPJ_BASICO", StringType(), True),
        StructField("CNPJ_ORDEM", StringType(), True),
        StructField("CNPJ_DV", StringType(), True),
        StructField("IDENTIFICADOR_MATRIZ_FILIAL", StringType(), True),
        StructField("NOME_FANTASIA", StringType(), True),
        StructField("SITUACAO_CADASTRAL", StringType(), True),
        StructField("DATA_SITUACAO_CADASTRAL", StringType(), True),
        StructField("MOTIVO_SITUACAO_CADASTRAL", StringType(), True),
        StructField("NOME_CIDADE_EXTERIOR", StringType(), True),
        StructField("PAIS", StringType(), True),
        StructField("DATA_INICIO_ATIVIDADE", StringType(), True),
        StructField("CNAE_PRINCIPAL", StringType(), True),
        StructField("CNAE_SECUNDARIO", StringType(), True),
        StructField("TIPO_LOGRADOURO", StringType(), True),
        StructField("LOGRADOURO", StringType(), True),
        StructField("NUMERO", StringType(), True),
        StructField("COMPLEMENTO", StringType(), True),
        StructField("BAIRRO", StringType(), True),
        StructField("CEP", StringType(), True),
        StructField("UF", StringType(), True),
        StructField("MUNICIPIO", StringType(), True),
        StructField("DDD1", StringType(), True),
        StructField("TELEFONE1", StringType(), True),
        StructField("DDD2", StringType(), True),
        StructField("TELEFONE2", StringType(), True),
        StructField("DDD_FAX", StringType(), True),
        StructField("FAX", StringType(), True),
        StructField("EMAIL", StringType(), True),
        StructField("SITUACAO_ESPECIAL", StringType(), True),
        StructField("DATA_SITUACAO_ESPECIAL", StringType(), True),
    ]),
    "Socios": StructType([
        StructField("CNPJ_BASICO", StringType(), True),
        StructField("IDENTIFICADOR_SOCIO", StringType(), True),
        StructField("NOME_SOCIO", StringType(), True),
        StructField("CNPJ_CPF_SOCIO", StringType(), True),
        StructField("QUALIFICACAO_SOCIO", StringType(), True),
        StructField("DATA_ENTRADA_SOCIEDADE", StringType(), True),
        StructField("PAIS", StringType(), True),
        StructField("REPRESENTANTE_LEGAL", StringType(), True),
        StructField("NOME_REPRESENTANTE", StringType(), True),
        StructField("QUALIFICACAO_REPRESENTANTE", StringType(), True),
        StructField("FAIXA_ETARIA", StringType(), True),
    ]),
    "Municipios": StructType([
        StructField("CODIGO", StringType(), True),
        StructField("DESCRICAO", StringType(), True),
    ]),
    "Naturezas": StructType([
        StructField("CODIGO", StringType(), True),
        StructField("DESCRICAO", StringType(), True),
    ]),
}

TABLE_NAMES = {
    "Empresas": "cnpj_empresas",
    "Estabelecimentos": "cnpj_estabelecimentos",
    "Socios": "cnpj_socios",
    "Municipios": "cnpj_municipios",
    "Naturezas": "cnpj_naturezas",
}

print(f"Schemas preparados para {len(SCHEMAS)} entidades.")

Schemas preparados para 5 entidades.


In [0]:
# Reconstitui o caminho EXTRACTED ativo quando necessário.
if "ACTIVE_EXTRACTED_ROOT" not in globals():
    if "EXTRACTED_ROOT" not in globals() or "PERIODO_REFERENCIA" not in globals():
        raise RuntimeError("Execute as seções 1 a 4 antes da carga Delta.")
    ACTIVE_EXTRACTED_ROOT = f"{EXTRACTED_ROOT}/{PERIODO_REFERENCIA}"

# Mantém o paralelismo do Spark. Não use repartition(1) nas bases grandes.
# No Databricks Serverless, as otimizações do Spark são gerenciadas automaticamente.

def extracted_files(entity: str) -> list[str]:
    directory = Path(f"{ACTIVE_EXTRACTED_ROOT}/{entity}")
    if not directory.exists():
        return []

    return [
        str(path)
        for path in directory.rglob("*")
        if path.is_file()
    ]


def extracted_files(entity: str) -> list[str]:
    directory = Path(f"{ACTIVE_EXTRACTED_ROOT}/{entity}")
    if not directory.exists():
        return []
    return [str(path) for path in directory.rglob("*") if path.is_file()]


def latest_output_rows(table_name: str) -> str:
    """Obtém numOutputRows do histórico Delta sem fazer um count completo."""
    try:
        history = spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 1").first()
        metrics = history["operationMetrics"] or {}
        return metrics.get("numOutputRows", "não informado")
    except Exception:
        return "não informado"


def load_entity_to_delta(entity: str) -> dict[str, str]:
    files = extracted_files(entity)
    if not files:
        return {"entidade": entity, "status": "sem arquivos", "tabela": "", "linhas": "0"}

    table_name = f"{CATALOG}.{SCHEMA}.{TABLE_NAMES[entity]}"
    input_path = f"{ACTIVE_EXTRACTED_ROOT}/{entity}"

    print("\n" + "=" * 90)
    print(f"Carregando {entity} -> {table_name}")
    print(f"Arquivos de entrada: {len(files)}")
    print("=" * 90)

    dataframe = (
        spark.read
        .option("delimiter", ";")
        .option("quote", '"')
        .option("header", "false")
        .option("encoding", "ISO-8859-1")
        .option("mode", "PERMISSIVE")
        .schema(SCHEMAS[entity])
        .csv(input_path)
        .withColumn("_arquivo_origem", F.col("_metadata.file_path"))
        .withColumn("_periodo_referencia", F.lit(PERIODO_REFERENCIA))
        .withColumn("_dt_ingestao", F.current_timestamp())
    )

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    rows = latest_output_rows(table_name)
    print(f"[OK] {table_name} carregada. Linhas gravadas: {rows}")

    return {
        "entidade": entity,
        "status": "carregada",
        "tabela": table_name,
        "linhas": str(rows),
    }


def run_delta_load() -> list[dict[str, str]]:
    if not LOAD_DELTA:
        print("[INFO] carregar_delta=false. A carga Delta foi ignorada.")
        return []

    results = []
    for entity in TARGET_ENTITIES:
        results.append(load_entity_to_delta(entity))
    return results


DELTA_RESULTS = run_delta_load()


Carregando Empresas -> workspace.default.cnpj_empresas
Arquivos de entrada: 10
[OK] workspace.default.cnpj_empresas carregada. Linhas gravadas: 69062850

Carregando Estabelecimentos -> workspace.default.cnpj_estabelecimentos
Arquivos de entrada: 10
[OK] workspace.default.cnpj_estabelecimentos carregada. Linhas gravadas: 72318975

Carregando Socios -> workspace.default.cnpj_socios
Arquivos de entrada: 10
[OK] workspace.default.cnpj_socios carregada. Linhas gravadas: 27992378

Carregando Municipios -> workspace.default.cnpj_municipios
Arquivos de entrada: 1
[OK] workspace.default.cnpj_municipios carregada. Linhas gravadas: 5572

Carregando Naturezas -> workspace.default.cnpj_naturezas
Arquivos de entrada: 1
[OK] workspace.default.cnpj_naturezas carregada. Linhas gravadas: 91


In [0]:
if DELTA_RESULTS:
    summary_df = spark.createDataFrame(DELTA_RESULTS)
    display(summary_df.select("entidade", "status", "tabela", "linhas"))

    loaded_tables = [item["tabela"] for item in DELTA_RESULTS if item["status"] == "carregada"]
    print("\nTabelas disponíveis:")
    for table in loaded_tables:
        print(f"- {table}")
else:
    print("Nenhuma tabela Delta foi carregada nesta execução.")

print(f"\nArquivos do pipeline: {PROJECT_ROOT}")
print(f"Período de referência: {PERIODO_REFERENCIA}")

entidade,status,tabela,linhas
Empresas,carregada,workspace.default.cnpj_empresas,69062850
Estabelecimentos,carregada,workspace.default.cnpj_estabelecimentos,72318975
Socios,carregada,workspace.default.cnpj_socios,27992378
Municipios,carregada,workspace.default.cnpj_municipios,5572
Naturezas,carregada,workspace.default.cnpj_naturezas,91



Tabelas disponíveis:
- workspace.default.cnpj_empresas
- workspace.default.cnpj_estabelecimentos
- workspace.default.cnpj_socios
- workspace.default.cnpj_municipios
- workspace.default.cnpj_naturezas

Arquivos do pipeline: /Volumes/workspace/default/cnpj_brasil/CNPJ_BRASIL
Período de referência: 2026-07


In [0]:
CATALOG = dbutils.widgets.get("catalogo")
SCHEMA = dbutils.widgets.get("schema")

In [0]:
display(SCHEMA)

'default'

In [0]:
%sql

SELECT
  e.UF,
  COUNT(*) AS QUANTIDADE
FROM workspace.default.cnpj_estabelecimentos e
WHERE e.SITUACAO_CADASTRAL = '02'
GROUP BY e.UF
ORDER BY QUANTIDADE DESC;



UF,QUANTIDADE
SP,8564328
MG,2889352
RJ,2242284
PR,1976120
RS,1734719
SC,1547618
BA,1243547
GO,1025711
PE,761039
CE,738330


In [0]:
%sql

SELECT *
FROM workspace.default.cnpj_estabelecimentos_ativos
LIMIT 100;

CNPJ,RAZAO_SOCIAL,NOME_FANTASIA,UF,MUNICIPIO,CNAE_PRINCIPAL
08527041000142,NAIR RECHI DA SILVA,null,SP,7097,0121101
08557163000181,FLORIANO FERREIRA DE CARVALHO FILHO,null,SP,6917,0113000
08920628000117,MARCIA CRISTINA LOPES SCIORTINO,MC AUTO PECAS,RS,8531,4530703
08957263000103,AVLUM SA,null,EX,9707,6462000
09160335000141,NEUSA APARECIDA GONCALVES BARROS,null,SP,6799,0151201
09242135000138,V8 SERVICOS DE COMINICACAO LTDA,V8 COMUNICACAO,SP,7107,5813100
09256014000145,NJJ GLOBAL IMPORT LTDA,null,SP,7107,1091102
09325136000146,PEDRO PAULO BEDRAN DE CASTRO,null,SP,6443,0113000
09376307000166,ELISANGELA MARIA SOUSA LUZ,null,CE,1389,9602502
09379881000178,VERITAX CONSULTORIA E GESTAO EMPRESARIAL LTDA,VERITAX GESTAO E RESULTADOS,PR,5453,8211300


In [0]:
%sql

SELECT COUNT(*) AS quantidade_registros
FROM workspace.default.cnpj_estabelecimentos_ativos;

quantidade_registros
27800284


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.cnpj_estabelecimentos_ativos
USING DELTA
AS

SELECT
  CONCAT(e.CNPJ_BASICO, e.CNPJ_ORDEM, e.CNPJ_DV) AS CNPJ,
  emp.RAZAO_SOCIAL,
  e.NOME_FANTASIA,
  e.UF,
  e.MUNICIPIO,
  e.CNAE_PRINCIPAL
FROM workspace.default.cnpj_estabelecimentos e
LEFT JOIN workspace.default.cnpj_empresas emp
  ON e.CNPJ_BASICO = emp.CNPJ_BASICO
WHERE e.SITUACAO_CADASTRAL = '02';

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Junção entre estabelecimento e razão social
SELECT
  CONCAT(e.CNPJ_BASICO, e.CNPJ_ORDEM, e.CNPJ_DV) AS CNPJ,
  emp.RAZAO_SOCIAL,
  e.NOME_FANTASIA,
  e.UF,
  e.MUNICIPIO,
  e.CNAE_PRINCIPAL
FROM workspace.default.cnpj_estabelecimentos e
LEFT JOIN workspace.default.cnpj_empresas emp
  ON e.CNPJ_BASICO = emp.CNPJ_BASICO
WHERE e.SITUACAO_CADASTRAL = '02';

CNPJ,RAZAO_SOCIAL,NOME_FANTASIA,UF,MUNICIPIO,CNAE_PRINCIPAL
43071267000182,RODOLFO MACHADO COQUE,null,SP,6775,0116401
43082238000116,43.082.238 GENI BRITO DIAS MOITINHO,null,SP,6545,1412602
43084489000130,BRAGANCA ASSESSORIA EMPRESARIAL E SERVICOS LTDA,BRAGANCA COMERCIO E SERVICOS,SP,7107,8111700
43155272000173,JANAINA CHAVES DESPACHANTE LTDA,DESPACHANTE BELTRAO,PR,7565,8299799
43158335000145,BRASIL SAT TECNOLOGIA EM SATELITE LTDA,BRASIL SAT TECNOLOGIA EM SATELITE,DF,9701,8020001
43174850000119,43.174.850 DENIS GONCALVES VALERIO,null,ES,5629,4399103
43178344000106,MARINA PIVA ARQUITETURA E URBANISMO LTDA,null,SC,8047,7119703
43182522000164,DIVINO MARTINS 08772010649,null,MG,5219,4781400
43183438000165,RODRIGO FERNANDO CUNHA DE PAULA 43971220835,null,SP,7107,9609206
43208557000125,NEIVA MURARO MINATTI 92769110900,null,SC,8225,4781400


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_empresa_publica
USING DELTA
AS

WITH empresas_norm AS (

    SELECT
        LPAD(
            REGEXP_REPLACE(TRIM(CNPJ_BASICO), '[^0-9]', ''),
            8,
            '0'
        ) AS CNPJ_BASICO,

        NULLIF(TRIM(RAZAO_SOCIAL), '') AS RAZAO_SOCIAL,

        LPAD(
            REGEXP_REPLACE(TRIM(NATUREZA_JURIDICA), '[^0-9]', ''),
            4,
            '0'
        ) AS CODIGO_NATUREZA_JURIDICA,

        LPAD(
            REGEXP_REPLACE(
                TRIM(QUALIFICACAO_RESPONSAVEL),
                '[^0-9]',
                ''
            ),
            2,
            '0'
        ) AS CODIGO_QUALIFICACAO_RESPONSAVEL,

        TRY_CAST(
            REPLACE(
                NULLIF(TRIM(CAPITAL_SOCIAL), ''),
                ',',
                '.'
            ) AS DECIMAL(18,2)
        ) AS CAPITAL_SOCIAL,

        LPAD(
            REGEXP_REPLACE(TRIM(PORTE_EMPRESA), '[^0-9]', ''),
            2,
            '0'
        ) AS CODIGO_PORTE_EMPRESA,

        NULLIF(TRIM(ENTE_FEDERATIVO), '') AS ENTE_FEDERATIVO

    FROM workspace.default.cnpj_empresas
),

estabelecimentos_norm AS (

    SELECT
        LPAD(
            REGEXP_REPLACE(TRIM(CNPJ_BASICO), '[^0-9]', ''),
            8,
            '0'
        ) AS CNPJ_BASICO,

        LPAD(
            REGEXP_REPLACE(TRIM(CNPJ_ORDEM), '[^0-9]', ''),
            4,
            '0'
        ) AS CNPJ_ORDEM,

        LPAD(
            REGEXP_REPLACE(TRIM(CNPJ_DV), '[^0-9]', ''),
            2,
            '0'
        ) AS CNPJ_DV,

        TRIM(IDENTIFICADOR_MATRIZ_FILIAL)
            AS IDENTIFICADOR_MATRIZ_FILIAL,

        NULLIF(TRIM(NOME_FANTASIA), '') AS NOME_FANTASIA,

        TRIM(SITUACAO_CADASTRAL) AS SITUACAO_CADASTRAL,

        CAST(
            TRY_TO_TIMESTAMP(
                NULLIF(TRIM(DATA_SITUACAO_CADASTRAL), ''),
                'yyyyMMdd'
            ) AS DATE
        ) AS DATA_SITUACAO_CADASTRAL,

        NULLIF(
            TRIM(MOTIVO_SITUACAO_CADASTRAL),
            ''
        ) AS MOTIVO_SITUACAO_CADASTRAL,

        CAST(
            TRY_TO_TIMESTAMP(
                NULLIF(TRIM(DATA_INICIO_ATIVIDADE), ''),
                'yyyyMMdd'
            ) AS DATE
        ) AS DATA_INICIO_ATIVIDADE,

        LPAD(
            REGEXP_REPLACE(TRIM(CNAE_PRINCIPAL), '[^0-9]', ''),
            7,
            '0'
        ) AS CNAE_PRINCIPAL,

        NULLIF(TRIM(CNAE_SECUNDARIO), '') AS CNAES_SECUNDARIOS,

        SIZE(
            ARRAY_DISTINCT(
                FILTER(
                    TRANSFORM(
                        SPLIT(
                            COALESCE(
                                NULLIF(TRIM(CNAE_SECUNDARIO), ''),
                                ''
                            ),
                            ','
                        ),
                        item -> TRIM(item)
                    ),
                    item -> item <> ''
                )
            )
        ) AS QTD_CNAES_SECUNDARIOS,

        NULLIF(TRIM(TIPO_LOGRADOURO), '') AS TIPO_LOGRADOURO,
        NULLIF(TRIM(LOGRADOURO), '') AS LOGRADOURO,
        NULLIF(TRIM(NUMERO), '') AS NUMERO,
        NULLIF(TRIM(COMPLEMENTO), '') AS COMPLEMENTO,
        NULLIF(TRIM(BAIRRO), '') AS BAIRRO,

        LPAD(
            REGEXP_REPLACE(TRIM(CEP), '[^0-9]', ''),
            8,
            '0'
        ) AS CEP,

        UPPER(NULLIF(TRIM(UF), '')) AS UF,

        LPAD(
            REGEXP_REPLACE(TRIM(MUNICIPIO), '[^0-9]', ''),
            4,
            '0'
        ) AS CODIGO_MUNICIPIO,

        NULLIF(
            REGEXP_REPLACE(
                COALESCE(TRIM(DDD1), ''),
                '[^0-9]',
                ''
            ),
            ''
        ) AS DDD_TELEFONE_1,

        NULLIF(
            REGEXP_REPLACE(
                COALESCE(TRIM(TELEFONE1), ''),
                '[^0-9]',
                ''
            ),
            ''
        ) AS NUMERO_TELEFONE_1,

        NULLIF(
            REGEXP_REPLACE(
                CONCAT(
                    COALESCE(TRIM(DDD1), ''),
                    COALESCE(TRIM(TELEFONE1), '')
                ),
                '[^0-9]',
                ''
            ),
            ''
        ) AS TELEFONE_1,

        NULLIF(
            REGEXP_REPLACE(
                COALESCE(TRIM(DDD2), ''),
                '[^0-9]',
                ''
            ),
            ''
        ) AS DDD_TELEFONE_2,

        NULLIF(
            REGEXP_REPLACE(
                COALESCE(TRIM(TELEFONE2), ''),
                '[^0-9]',
                ''
            ),
            ''
        ) AS NUMERO_TELEFONE_2,

        NULLIF(
            REGEXP_REPLACE(
                CONCAT(
                    COALESCE(TRIM(DDD2), ''),
                    COALESCE(TRIM(TELEFONE2), '')
                ),
                '[^0-9]',
                ''
            ),
            ''
        ) AS TELEFONE_2,

        NULLIF(
            REGEXP_REPLACE(
                CONCAT(
                    COALESCE(TRIM(DDD_FAX), ''),
                    COALESCE(TRIM(FAX), '')
                ),
                '[^0-9]',
                ''
            ),
            ''
        ) AS TELEFONE_FAX,

        LOWER(NULLIF(TRIM(EMAIL), '')) AS EMAIL,

        NULLIF(TRIM(SITUACAO_ESPECIAL), '')
            AS SITUACAO_ESPECIAL,

        CAST(
            TRY_TO_TIMESTAMP(
                NULLIF(TRIM(DATA_SITUACAO_ESPECIAL), ''),
                'yyyyMMdd'
            ) AS DATE
        ) AS DATA_SITUACAO_ESPECIAL,

        _periodo_referencia AS PERIODO_REFERENCIA,
        _dt_ingestao AS DATA_INGESTAO

    FROM workspace.default.cnpj_estabelecimentos

    WHERE TRIM(SITUACAO_CADASTRAL) = '02'
      AND TRIM(IDENTIFICADOR_MATRIZ_FILIAL) = '1'
),

socios_agg AS (

    SELECT
        LPAD(
            REGEXP_REPLACE(TRIM(CNPJ_BASICO), '[^0-9]', ''),
            8,
            '0'
        ) AS CNPJ_BASICO,

        COUNT(*) AS QTD_SOCIOS,

        /* Mantém a mesma classificação utilizada no notebook original. */
        SUM(
            CASE
                WHEN TRIM(IDENTIFICADOR_SOCIO) = '1' THEN 1
                ELSE 0
            END
        ) AS QTD_SOCIOS_PF,

        SUM(
            CASE
                WHEN TRIM(IDENTIFICADOR_SOCIO) = '2' THEN 1
                ELSE 0
            END
        ) AS QTD_SOCIOS_PJ,

        SUM(
            CASE
                WHEN TRIM(IDENTIFICADOR_SOCIO) = '3' THEN 1
                ELSE 0
            END
        ) AS QTD_SOCIOS_ESTRANGEIROS,

        MIN(
            CAST(
                TRY_TO_TIMESTAMP(
                    NULLIF(TRIM(DATA_ENTRADA_SOCIEDADE), ''),
                    'yyyyMMdd'
                ) AS DATE
            )
        ) AS DATA_ENTRADA_SOCIO_MAIS_ANTIGO,

        MAX(
            CAST(
                TRY_TO_TIMESTAMP(
                    NULLIF(TRIM(DATA_ENTRADA_SOCIEDADE), ''),
                    'yyyyMMdd'
                ) AS DATE
            )
        ) AS DATA_ENTRADA_SOCIO_MAIS_RECENTE

    FROM workspace.default.cnpj_socios

    GROUP BY
        LPAD(
            REGEXP_REPLACE(TRIM(CNPJ_BASICO), '[^0-9]', ''),
            8,
            '0'
        )
),

municipios_dim AS (

    SELECT
        LPAD(
            REGEXP_REPLACE(TRIM(CODIGO), '[^0-9]', ''),
            4,
            '0'
        ) AS CODIGO_MUNICIPIO,

        MAX(NULLIF(TRIM(DESCRICAO), '')) AS NOME_MUNICIPIO

    FROM workspace.default.cnpj_municipios

    GROUP BY
        LPAD(
            REGEXP_REPLACE(TRIM(CODIGO), '[^0-9]', ''),
            4,
            '0'
        )
),

naturezas_dim AS (

    SELECT
        LPAD(
            REGEXP_REPLACE(TRIM(CODIGO), '[^0-9]', ''),
            4,
            '0'
        ) AS CODIGO_NATUREZA_JURIDICA,

        MAX(NULLIF(TRIM(DESCRICAO), ''))
            AS DESCRICAO_NATUREZA_JURIDICA

    FROM workspace.default.cnpj_naturezas

    GROUP BY
        LPAD(
            REGEXP_REPLACE(TRIM(CODIGO), '[^0-9]', ''),
            4,
            '0'
        )
),

base_enriquecida AS (

    SELECT
        CONCAT(
            est.CNPJ_BASICO,
            est.CNPJ_ORDEM,
            est.CNPJ_DV
        ) AS CNPJ,

        est.CNPJ_BASICO,
        est.CNPJ_ORDEM,
        est.CNPJ_DV,

        'MATRIZ' AS TIPO_ESTABELECIMENTO,

        est.SITUACAO_CADASTRAL,
        'ATIVA' AS DESCRICAO_SITUACAO_CADASTRAL,
        est.DATA_SITUACAO_CADASTRAL,
        est.MOTIVO_SITUACAO_CADASTRAL,

        emp.RAZAO_SOCIAL,
        est.NOME_FANTASIA,

        emp.CODIGO_NATUREZA_JURIDICA,
        nat.DESCRICAO_NATUREZA_JURIDICA,

        emp.CODIGO_QUALIFICACAO_RESPONSAVEL,
        emp.CODIGO_PORTE_EMPRESA,

        CASE emp.CODIGO_PORTE_EMPRESA
            WHEN '00' THEN 'NÃO INFORMADO'
            WHEN '01' THEN 'MICROEMPRESA'
            WHEN '03' THEN 'EMPRESA DE PEQUENO PORTE'
            WHEN '05' THEN 'DEMAIS'
            ELSE 'NÃO CLASSIFICADO'
        END AS DESCRICAO_PORTE_EMPRESA,

        emp.CAPITAL_SOCIAL,

        CASE
            WHEN emp.CAPITAL_SOCIAL IS NULL
                THEN 'NÃO INFORMADO'
            WHEN emp.CAPITAL_SOCIAL <= 10000
                THEN 'ATÉ 10 MIL'
            WHEN emp.CAPITAL_SOCIAL <= 50000
                THEN '10 A 50 MIL'
            WHEN emp.CAPITAL_SOCIAL <= 200000
                THEN '50 A 200 MIL'
            WHEN emp.CAPITAL_SOCIAL <= 1000000
                THEN '200 MIL A 1 MILHÃO'
            ELSE 'ACIMA DE 1 MILHÃO'
        END AS FAIXA_CAPITAL_SOCIAL,

        emp.ENTE_FEDERATIVO,

        est.DATA_INICIO_ATIVIDADE,

        CASE
            WHEN est.DATA_INICIO_ATIVIDADE IS NULL THEN NULL
            WHEN est.DATA_INICIO_ATIVIDADE > CURRENT_DATE() THEN NULL
            ELSE CAST(
                FLOOR(
                    MONTHS_BETWEEN(
                        CURRENT_DATE(),
                        est.DATA_INICIO_ATIVIDADE
                    ) / 12
                ) AS INT
            )
        END AS IDADE_EMPRESA_ANOS,

        est.CNAE_PRINCIPAL,
        LEFT(est.CNAE_PRINCIPAL, 2) AS DIVISAO_CNAE,
        est.CNAES_SECUNDARIOS,
        est.QTD_CNAES_SECUNDARIOS,

        CASE
            WHEN est.CNAE_PRINCIPAL IS NULL
                THEN est.QTD_CNAES_SECUNDARIOS
            ELSE est.QTD_CNAES_SECUNDARIOS + 1
        END AS QTD_CNAES_TOTAL,

        CASE
            WHEN est.QTD_CNAES_SECUNDARIOS > 0 THEN 1
            ELSE 0
        END AS FLAG_MULTICNAE,

        est.TIPO_LOGRADOURO,
        est.LOGRADOURO,
        est.NUMERO,
        est.COMPLEMENTO,
        est.BAIRRO,
        est.CEP,
        est.CODIGO_MUNICIPIO,
        mun.NOME_MUNICIPIO,
        est.UF,

        NULLIF(
            TRIM(
                CONCAT_WS(
                    ' ',
                    est.TIPO_LOGRADOURO,
                    est.LOGRADOURO
                )
            ),
            ''
        ) AS ENDERECO_LOGRADOURO,

        CONCAT_WS(
            ', ',
            NULLIF(
                TRIM(
                    CONCAT_WS(
                        ' ',
                        est.TIPO_LOGRADOURO,
                        est.LOGRADOURO
                    )
                ),
                ''
            ),
            est.NUMERO,
            est.COMPLEMENTO,
            est.BAIRRO
        ) AS ENDERECO,

        CONCAT_WS(
            ', ',
            NULLIF(
                TRIM(
                    CONCAT_WS(
                        ' ',
                        est.TIPO_LOGRADOURO,
                        est.LOGRADOURO
                    )
                ),
                ''
            ),
            est.NUMERO,
            est.COMPLEMENTO,
            est.BAIRRO,
            mun.NOME_MUNICIPIO,
            est.UF,
            CASE
                WHEN est.CEP IS NOT NULL
                    THEN CONCAT('CEP ', est.CEP)
            END
        ) AS ENDERECO_COMPLETO,

        est.DDD_TELEFONE_1,
        est.NUMERO_TELEFONE_1,
        est.TELEFONE_1,

        est.DDD_TELEFONE_2,
        est.NUMERO_TELEFONE_2,
        est.TELEFONE_2,

        est.TELEFONE_FAX,

        COALESCE(
            est.TELEFONE_1,
            est.TELEFONE_2
        ) AS TELEFONE_PRINCIPAL,

        est.EMAIL,

        COALESCE(s.QTD_SOCIOS, 0) AS QTD_SOCIOS,
        COALESCE(s.QTD_SOCIOS_PF, 0) AS QTD_SOCIOS_PF,
        COALESCE(s.QTD_SOCIOS_PJ, 0) AS QTD_SOCIOS_PJ,

        COALESCE(
            s.QTD_SOCIOS_ESTRANGEIROS,
            0
        ) AS QTD_SOCIOS_ESTRANGEIROS,

        s.DATA_ENTRADA_SOCIO_MAIS_ANTIGO,
        s.DATA_ENTRADA_SOCIO_MAIS_RECENTE,

        est.SITUACAO_ESPECIAL,
        est.DATA_SITUACAO_ESPECIAL,

        est.PERIODO_REFERENCIA,
        est.DATA_INGESTAO

    FROM estabelecimentos_norm est

    LEFT JOIN empresas_norm emp
        ON est.CNPJ_BASICO = emp.CNPJ_BASICO

    LEFT JOIN socios_agg s
        ON est.CNPJ_BASICO = s.CNPJ_BASICO

    LEFT JOIN municipios_dim mun
        ON est.CODIGO_MUNICIPIO = mun.CODIGO_MUNICIPIO

    LEFT JOIN naturezas_dim nat
        ON emp.CODIGO_NATUREZA_JURIDICA =
           nat.CODIGO_NATUREZA_JURIDICA
)

SELECT
    *,

    CASE
        WHEN IDADE_EMPRESA_ANOS IS NULL
            THEN 'NÃO INFORMADO'
        WHEN IDADE_EMPRESA_ANOS < 1
            THEN 'MENOS DE 1 ANO'
        WHEN IDADE_EMPRESA_ANOS <= 2
            THEN '1 A 2 ANOS'
        WHEN IDADE_EMPRESA_ANOS <= 5
            THEN '3 A 5 ANOS'
        WHEN IDADE_EMPRESA_ANOS <= 10
            THEN '6 A 10 ANOS'
        WHEN IDADE_EMPRESA_ANOS <= 20
            THEN '11 A 20 ANOS'
        ELSE '21 ANOS OU MAIS'
    END AS FAIXA_IDADE_EMPRESA,

    CASE
        WHEN LENGTH(TELEFONE_PRINCIPAL) = 11 THEN
            CONCAT(
                '(',
                SUBSTRING(TELEFONE_PRINCIPAL, 1, 2),
                ') ',
                SUBSTRING(TELEFONE_PRINCIPAL, 3, 5),
                '-',
                SUBSTRING(TELEFONE_PRINCIPAL, 8, 4)
            )

        WHEN LENGTH(TELEFONE_PRINCIPAL) = 10 THEN
            CONCAT(
                '(',
                SUBSTRING(TELEFONE_PRINCIPAL, 1, 2),
                ') ',
                SUBSTRING(TELEFONE_PRINCIPAL, 3, 4),
                '-',
                SUBSTRING(TELEFONE_PRINCIPAL, 7, 4)
            )

        ELSE TELEFONE_PRINCIPAL
    END AS TELEFONE_PRINCIPAL_FORMATADO,

    CASE
        WHEN LENGTH(TELEFONE_PRINCIPAL) = 11
             AND SUBSTRING(TELEFONE_PRINCIPAL, 3, 1) = '9'
            THEN 'CELULAR'

        WHEN LENGTH(TELEFONE_PRINCIPAL) = 10
            THEN 'FIXO'

        WHEN TELEFONE_PRINCIPAL IS NULL
            THEN 'NÃO INFORMADO'

        ELSE 'NÃO CLASSIFICADO'
    END AS TIPO_TELEFONE_PRINCIPAL,

    CASE
        WHEN LENGTH(TELEFONE_PRINCIPAL) IN (10, 11) THEN 1
        ELSE 0
    END AS FLAG_TELEFONE_VALIDO,

    CASE
        WHEN TELEFONE_PRINCIPAL IS NOT NULL THEN 1
        ELSE 0
    END AS FLAG_POSSUI_TELEFONE,

    CASE
        WHEN EMAIL IS NOT NULL THEN 1
        ELSE 0
    END AS FLAG_POSSUI_EMAIL,

    CASE
        WHEN EMAIL RLIKE
             '^[a-z0-9._%+-]+@[a-z0-9.-]+\\.[a-z]{2,}$'
            THEN 1
        ELSE 0
    END AS FLAG_EMAIL_VALIDO,

    CASE
        WHEN LOGRADOURO IS NOT NULL
         AND NUMERO IS NOT NULL
         AND CEP IS NOT NULL
         AND NOME_MUNICIPIO IS NOT NULL
         AND UF IS NOT NULL
            THEN 1
        ELSE 0
    END AS FLAG_ENDERECO_COMPLETO,

    CASE
        WHEN TELEFONE_PRINCIPAL IS NOT NULL
          OR EMAIL IS NOT NULL
            THEN 1
        ELSE 0
    END AS FLAG_CONTATO_DISPONIVEL,

    CASE
        WHEN QTD_SOCIOS_PJ > 0 THEN 1
        ELSE 0
    END AS FLAG_POSSUI_SOCIO_PJ,

    CURRENT_TIMESTAMP() AS DATA_PROCESSAMENTO

FROM base_enriquecida;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS TOTAL_LINHAS,
    COUNT(DISTINCT CNPJ) AS TOTAL_CNPJS,
    COUNT(*) - COUNT(DISTINCT CNPJ) AS CNPJS_DUPLICADOS
FROM workspace.default.gold_empresa_publica;

TOTAL_LINHAS,TOTAL_CNPJS,CNPJS_DUPLICADOS
26486758,26486756,2


In [0]:
%sql

SELECT
    COUNT(*) AS TOTAL_EMPRESAS,

    SUM(
        CASE WHEN NOME_MUNICIPIO IS NOT NULL THEN 1 ELSE 0 END
    ) AS COM_MUNICIPIO,

    SUM(
        CASE WHEN DESCRICAO_NATUREZA_JURIDICA IS NOT NULL
             THEN 1 ELSE 0 END
    ) AS COM_NATUREZA_JURIDICA,

    SUM(
        CASE WHEN QTD_SOCIOS > 0 THEN 1 ELSE 0 END
    ) AS COM_SOCIOS,

    ROUND(
        100.0 * AVG(
            CASE WHEN NOME_MUNICIPIO IS NOT NULL THEN 1 ELSE 0 END
        ),
        2
    ) AS PCT_COM_MUNICIPIO,

    ROUND(
        100.0 * AVG(
            CASE WHEN QTD_SOCIOS > 0 THEN 1 ELSE 0 END
        ),
        2
    ) AS PCT_COM_SOCIOS

FROM workspace.default.gold_empresa_publica;

TOTAL_EMPRESAS,COM_MUNICIPIO,COM_NATUREZA_JURIDICA,COM_SOCIOS,PCT_COM_MUNICIPIO,PCT_COM_SOCIOS
26486758,26486748,26486757,8912208,100.0,33.65


In [0]:
%sql

SELECT
    *
FROM workspace.default.gold_empresa_publica
LIMIT 100;

CNPJ,CNPJ_BASICO,CNPJ_ORDEM,CNPJ_DV,TIPO_ESTABELECIMENTO,SITUACAO_CADASTRAL,DESCRICAO_SITUACAO_CADASTRAL,DATA_SITUACAO_CADASTRAL,MOTIVO_SITUACAO_CADASTRAL,RAZAO_SOCIAL,NOME_FANTASIA,CODIGO_NATUREZA_JURIDICA,DESCRICAO_NATUREZA_JURIDICA,CODIGO_QUALIFICACAO_RESPONSAVEL,CODIGO_PORTE_EMPRESA,DESCRICAO_PORTE_EMPRESA,CAPITAL_SOCIAL,FAIXA_CAPITAL_SOCIAL,ENTE_FEDERATIVO,DATA_INICIO_ATIVIDADE,IDADE_EMPRESA_ANOS,CNAE_PRINCIPAL,DIVISAO_CNAE,CNAES_SECUNDARIOS,QTD_CNAES_SECUNDARIOS,QTD_CNAES_TOTAL,FLAG_MULTICNAE,TIPO_LOGRADOURO,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CEP,CODIGO_MUNICIPIO,NOME_MUNICIPIO,UF,ENDERECO_LOGRADOURO,ENDERECO,ENDERECO_COMPLETO,DDD_TELEFONE_1,NUMERO_TELEFONE_1,TELEFONE_1,DDD_TELEFONE_2,NUMERO_TELEFONE_2,TELEFONE_2,TELEFONE_FAX,TELEFONE_PRINCIPAL,EMAIL,QTD_SOCIOS,QTD_SOCIOS_PF,QTD_SOCIOS_PJ,QTD_SOCIOS_ESTRANGEIROS,DATA_ENTRADA_SOCIO_MAIS_ANTIGO,DATA_ENTRADA_SOCIO_MAIS_RECENTE,SITUACAO_ESPECIAL,DATA_SITUACAO_ESPECIAL,PERIODO_REFERENCIA,DATA_INGESTAO,FAIXA_IDADE_EMPRESA,TELEFONE_PRINCIPAL_FORMATADO,TIPO_TELEFONE_PRINCIPAL,FLAG_TELEFONE_VALIDO,FLAG_POSSUI_TELEFONE,FLAG_POSSUI_EMAIL,FLAG_EMAIL_VALIDO,FLAG_ENDERECO_COMPLETO,FLAG_CONTATO_DISPONIVEL,FLAG_POSSUI_SOCIO_PJ,DATA_PROCESSAMENTO
01141561000173,01141561,0001,73,MATRIZ,02,ATIVA,2025-08-14,00,ANTONIO SERGIO TRENTIM,TRENTIM MATERIAIS DE CONSTRUCAO,2135,Empresário (Individual),50,01,MICROEMPRESA,30000.00,10 A 50 MIL,null,1996-04-01,30,4744099,47,null,0,1,0,RUA,BENTO DE ABREU,646,null,CENTRO,14825000,7039,SANTA LUCIA,SP,RUA BENTO DE ABREU,"RUA BENTO DE ABREU, 646, CENTRO","RUA BENTO DE ABREU, 646, CENTRO, SANTA LUCIA, SP, CEP 14825000",16,30106422,1630106422,null,null,null,null,1630106422,geraldogmcontabil@gmail.com,0,0,0,0,null,null,null,null,2026-07,2026-08-03T21:47:38.595Z,21 ANOS OU MAIS,(16) 3010-6422,FIXO,1,1,1,1,1,1,0,2026-08-05T01:10:46.914Z
01153871000108,01153871,0001,08,MATRIZ,02,ATIVA,1998-07-28,00,FUNDO MUNICIPAL DE ASSISTENCIA SOCIAL DO MUNICIPIO DE LONTRAS,FUNDO DE ASSISTENCIA SOCIAL DO MUNICIPIO DE LONTRAS,1333,Fundo Público da Administração Direta Municipal,05,05,DEMAIS,0.00,ATÉ 10 MIL,LONTRAS - SC,1996-04-22,30,8800600,88,null,0,1,0,PRACA,HENRIQUE SCHROEDER,01,null,CENTRO,89182000,8195,LONTRAS,SC,PRACA HENRIQUE SCHROEDER,"PRACA HENRIQUE SCHROEDER, 01, CENTRO","PRACA HENRIQUE SCHROEDER, 01, CENTRO, LONTRAS, SC, CEP 89182000",null,null,null,null,null,null,null,null,null,0,0,0,0,null,null,null,null,2026-07,2026-08-03T21:47:38.595Z,21 ANOS OU MAIS,null,NÃO INFORMADO,0,0,0,0,1,0,0,2026-08-05T01:10:46.914Z
01161687000100,01161687,0001,00,MATRIZ,02,ATIVA,2005-11-03,00,PANIFICADORA E MERCEARIA DA HORA LTDA,null,2062,Sociedade Empresária Limitada,49,01,MICROEMPRESA,0.00,ATÉ 10 MIL,null,1996-04-18,30,4721102,47,null,0,1,0,AVENIDA,DR. JOAO BATISTA DE SANTANA,1853,null,CENTRO,14790000,6449,GUAIRA,SP,AVENIDA DR. JOAO BATISTA DE SANTANA,"AVENIDA DR. JOAO BATISTA DE SANTANA, 1853, CENTRO","AVENIDA DR. JOAO BATISTA DE SANTANA, 1853, CENTRO, GUAIRA, SP, CEP 14790000",017,33314585,01733314585,null,null,null,017333316363,01733314585,null,0,0,0,0,null,null,null,null,2026-07,2026-08-03T21:47:38.595Z,21 ANOS OU MAIS,(01) 73331-4585,NÃO CLASSIFICADO,1,1,0,0,1,1,0,2026-08-05T01:10:46.914Z
73989972000124,73989972,0001,24,MATRIZ,02,ATIVA,2025-05-02,00,ASSOCIACAO DOS MORADORES DA RUA PEDRO SIMON,null,3999,Associação Privada,16,05,DEMAIS,0.00,ATÉ 10 MIL,null,1993-12-28,32,9430800,94,"9493600,9499500",2,3,1,RUA,PEDRO SIMON,SN,null,MARGEM ESQUERDA,89110001,8117,GASPAR,SC,RUA PEDRO SIMON,"RUA PEDRO SIMON, SN, MARGEM ESQUERDA","RUA PEDRO SIMON, SN, MARGEM ESQUERDA, GASPAR, SC, CEP 89110001",null,null,null,null,null,null,null,null,null,1,0,1,0,2013-06-18,2013-06-18,null,null,2026-07,2026-08-03T21:47:38.595Z,21 ANOS OU MAIS,null,NÃO INFORMADO,0,0,0,0,1,0,1,2026-08-05T01:10:46.914Z
06236679000107,06236679,0001,07,MATRIZ,02,ATIVA,2005-02-05,00,TECNOWOLF COMERCIO LTDA,null,2062,Sociedade Empresária Limitada,49,03,EMPRESA DE PEQUENO PORTE,100000.00,50 A 200 MIL,null,2004-04-20,22,4753900,47,"3

In [0]:
%sql

SELECT
    COUNT(*) AS TOTAL_EMPRESAS,
    COUNT(DISTINCT CNPJ) AS TOTAL_CNPJS,
    COUNT(*) - COUNT(DISTINCT CNPJ) AS CNPJS_DUPLICADOS,

    SUM(FLAG_POSSUI_TELEFONE) AS COM_TELEFONE,
    SUM(FLAG_TELEFONE_VALIDO) AS COM_TELEFONE_VALIDO,
    SUM(FLAG_POSSUI_EMAIL) AS COM_EMAIL,
    SUM(FLAG_EMAIL_VALIDO) AS COM_EMAIL_VALIDO,
    SUM(FLAG_ENDERECO_COMPLETO) AS COM_ENDERECO_COMPLETO,

    ROUND(
        100.0 * AVG(FLAG_POSSUI_TELEFONE),
        2
    ) AS PCT_COM_TELEFONE,

    ROUND(
        100.0 * AVG(FLAG_POSSUI_EMAIL),
        2
    ) AS PCT_COM_EMAIL,

    ROUND(
        100.0 * AVG(FLAG_ENDERECO_COMPLETO),
        2
    ) AS PCT_ENDERECO_COMPLETO

FROM workspace.default.gold_empresa_publica;

TOTAL_EMPRESAS,TOTAL_CNPJS,CNPJS_DUPLICADOS,COM_TELEFONE,COM_TELEFONE_VALIDO,COM_EMAIL,COM_EMAIL_VALIDO,COM_ENDERECO_COMPLETO,PCT_COM_TELEFONE,PCT_COM_EMAIL,PCT_ENDERECO_COMPLETO
26486758,26486756,2,25793597,25562163,23866382,23844206,26411190,97.38,90.11,99.71
